# AI-Based Pothole Recognition Using UAVs
## Visual Inspection & Annotation Verification Notebook (DA1 Milestone)

**Project:** Automated Road Inspection Using Aerial Drone Imagery, CLAHE, and YOLO  
**Authors:** Ajay Kumaar (24BRS1287) & Hariaswath (24BRS1290)  
**Course:** BCSE306L – Artificial Intelligence  

---
### Purpose of this Notebook
1. Validate and visually inspect ground-truth YOLO annotations (`.txt`) against UAV high-resolution aerial road images.
2. Overlay color-coded bounding boxes for both classes:
   - **Class 0:** `agujero` (Pothole / Depression) — Displayed in **Red / Coral**
   - **Class 1:** `grietas` (Crack / Surface Fissure) — Displayed in **Cyan / Sky Blue**
3. Compare **Raw Aerial Imagery vs. CIE LAB-Space CLAHE Preprocessed Imagery** to visually evaluate shadow attenuation and contrast enhancement on asphalt surfaces.

In [ ]:
# Cell 1: Environment Setup and Library Imports
import os
import sys
import random
from pathlib import Path
from typing import List, Tuple, Dict

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Set high plotting resolution and clean aesthetics
%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# Add project root to sys.path to access internal data modules
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"[INFO] Project root: {PROJECT_ROOT}")
print(f"[INFO] OpenCV version: {cv2.__version__}")

---  
### 2. Dataset Discovery and Pair Matching
We locate the dataset directory, identify all matching `.png` and `.txt` pairs, and define class color assignments.

In [ ]:
# Cell 2: Configure Class Definitions and Path Auto-Discovery
CLASS_MAP = {
    0: "agujero (Pothole)",
    1: "grietas (Crack)"
}

# Color map for visual overlays (RGB format)
COLOR_MAP = {
    0: (230, 57, 70),     # Crimson Red for Potholes
    1: (0, 180, 216)     # Bright Cyan for Cracks
}

# Search candidate dataset directories
candidate_dirs = [
    PROJECT_ROOT / "dataset",
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT / "preprocessing" / "output",
    PROJECT_ROOT / "preprocessing" / "output_resized"
]

dataset_dir = None
for c_dir in candidate_dirs:
    if c_dir.exists() and any(c_dir.glob("*.*")):
        dataset_dir = c_dir
        break

if dataset_dir:
    print(f"[SUCCESS] Active dataset directory found: {dataset_dir}")
else:
    print(f"[NOTE] Raw dataset directory not yet populated. Using fallback/simulated inspection.")
    dataset_dir = PROJECT_ROOT / "dataset"

---  
### 3. Coordinate Decoding & CLAHE Enhancement Functions
YOLO coordinates are normalized: `[class_id, x_center, y_center, width, height]` in $[0, 1]$.  
We convert these normalized ratios to absolute pixel bounding boxes: $[x_{\text{min}}, y_{\text{min}}, x_{\text{max}}, y_{\text{max}}]$.  
We also define the **CIE LAB CLAHE** enhancement function on the $L^*$ luminance channel.

In [ ]:
# Cell 3: Coordinate Conversion and LAB-CLAHE Preprocessing Logic

def parse_yolo_labels(label_path: Path, img_width: int, img_height: int) -> List[Dict]:
    """Parse YOLO label file and return list of bounding box dicts in pixel coordinates."""
    boxes = []
    if not label_path.exists():
        return boxes
        
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                x_c, y_c, w, h = map(float, parts[1:5])
                
                # Denormalize to pixel coordinates
                x_min = int((x_c - w / 2) * img_width)
                y_min = int((y_c - h / 2) * img_height)
                x_max = int((x_c + w / 2) * img_width)
                y_max = int((y_c + h / 2) * img_height)
                
                # Clamp to image boundaries
                x_min = max(0, min(img_width - 1, x_min))
                y_min = max(0, min(img_height - 1, y_min))
                x_max = max(0, min(img_width - 1, x_max))
                y_max = max(0, min(img_height - 1, y_max))
                
                boxes.append({
                    "class_id": cls_id,
                    "class_name": CLASS_MAP.get(cls_id, f"Class {cls_id}"),
                    "bbox": (x_min, y_min, x_max, y_max),
                    "norm_wh": (w, h),
                    "area_norm": w * h
                })
    return boxes


def apply_clahe_lab(image_bgr: np.ndarray, clip_limit: float = 2.0, grid_size: int = 8) -> np.ndarray:
    """Apply CLAHE strictly to L* channel in CIE LAB color space to avoid color distortion."""
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(grid_size, grid_size))
    l_eq = clahe.apply(l)
    enhanced_lab = cv2.merge([l_eq, a, b])
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)


def draw_annotations(image_rgb: np.ndarray, boxes: List[Dict], line_thickness: int = 3) -> np.ndarray:
    """Draw clean, styled bounding boxes and category tags on the image."""
    annotated = image_rgb.copy()
    for item in boxes:
        cls_id = item["class_id"]
        x_min, y_min, x_max, y_max = item["bbox"]
        color = COLOR_MAP.get(cls_id, (255, 255, 0))
        
        # Draw rectangle
        cv2.rectangle(annotated, (x_min, y_min), (x_max, y_max), color, line_thickness)
        
        # Draw label badge
        label_text = f"{item['class_name'].split()[0]}"
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = max(0.5, line_thickness * 0.2)
        thickness = max(1, int(line_thickness * 0.4))
        (tw, th), baseline = cv2.getTextSize(label_text, font, font_scale, thickness)
        
        badge_y1 = max(0, y_min - th - 6)
        badge_y2 = y_min
        cv2.rectangle(annotated, (x_min, badge_y1), (x_min + tw + 8, badge_y2), color, -1)
        cv2.putText(annotated, label_text, (x_min + 4, y_min - 4), font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)
    return annotated

print("[INFO] Annotation parsing and drawing functions defined successfully.")

---  
### 4. Visual Inspection of 6 Random Samples with Ground-Truth Overlays
We randomly sample 6 matched image-label pairs and visualize the ground-truth pothole (`agujero`) and crack (`grietas`) bounding boxes.

In [ ]:
# Cell 4: Load and Display 6 Random Dataset Samples with Bounding Boxes
image_files = list(dataset_dir.glob("*.*")) if dataset_dir.exists() else []

if len(image_files) >= 6:
    sampled_images = random.sample(image_files, 6)
    fig, axes = plt.subplots(3, 2, figsize=(16, 20))
    axes = axes.flatten()
    
    for i, img_path in enumerate(sampled_images):
        lbl_path = img_path.with_suffix(".txt")
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_bgr.shape[:2]
        
        boxes = parse_yolo_labels(lbl_path, w, h)
        annotated = draw_annotations(img_rgb, boxes, line_thickness=max(2, int(w/600)))
        
        potholes = sum(1 for b in boxes if b['class_id'] == 0)
        cracks = sum(1 for b in boxes if b['class_id'] == 1)
        
        axes[i].imshow(annotated)
        axes[i].set_title(f"{img_path.name} | Potholes: {potholes}, Cracks: {cracks}", fontsize=12, fontweight='bold')
        axes[i].axis("off")
        
    plt.suptitle("UAV Aerial Pothole & Crack Ground-Truth Annotations (Sample of 6)", fontsize=16, fontweight='bold', y=0.99)
    plt.tight_layout()
    plt.show()
else:
    print(f"[NOTE] Found {len(image_files)} images in '{dataset_dir}'.")
    print("To visualize the actual images, ensure your raw dataset is placed inside 'dataset/' directory.")

---  
### 5. Side-by-Side Comparison: Raw vs. CLAHE Preprocessed
Here we display the same aerial road section before and after **CIE LAB CLAHE** to observe contrast expansion in shadowed road surfaces.

In [ ]:
# Cell 5: Raw vs. CLAHE Enhancement Comparison
if image_files:
    sample_path = image_files[0]
    lbl_path = sample_path.with_suffix(".txt")
    
    raw_bgr = cv2.imread(str(sample_path))
    raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
    h, w = raw_bgr.shape[:2]
    
    # Apply CLAHE
    clahe_bgr = apply_clahe_lab(raw_bgr, clip_limit=2.0, grid_size=8)
    clahe_rgb = cv2.cvtColor(clahe_bgr, cv2.COLOR_BGR2RGB)
    
    # Overlay labels
    boxes = parse_yolo_labels(lbl_path, w, h)
    raw_annotated = draw_annotations(raw_rgb, boxes, line_thickness=max(2, int(w/600)))
    clahe_annotated = draw_annotations(clahe_rgb, boxes, line_thickness=max(2, int(w/600)))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9))
    ax1.imshow(raw_annotated)
    ax1.set_title(f"Original UAV Image ({sample_path.name})", fontsize=14, fontweight='bold')
    ax1.axis("off")
    
    ax2.imshow(clahe_annotated)
    ax2.set_title("CIE LAB CLAHE Enhanced Image (L* Equalized, Clip=2.0)", fontsize=14, fontweight='bold')
    ax2.axis("off")
    
    plt.suptitle("Effect of Luminance Contrast Enhancement on UAV Road Surface Distress", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("[NOTE] Run this cell once images are placed in 'dataset/'.")

---  
### 6. Pothole Severity Distribution (Preliminary Inspection)
We calculate the normalized area ($w \times h$) for all ground-truth potholes in the sample and plot the preliminary severity distribution:
- **Minor:** $\text{Area} < 0.005$
- **Moderate:** $0.005 \le \text{Area} < 0.025$
- **Severe:** $\text{Area} \ge 0.025$

In [ ]:
# Cell 6: Bounding Box Area Severity Histogram
all_label_files = list(dataset_dir.glob("*.txt")) if dataset_dir.exists() else []
areas = []

for lbl_p in all_label_files:
    if lbl_p.name == "classes.txt":
        continue
    with open(lbl_p, "r") as f:
        for line in f:
            p = line.strip().split()
            if len(p) >= 5 and int(p[0]) == 0:  # Potholes only
                w, h = float(p[3]), float(p[4])
                areas.append(w * h)

if areas:
    areas = np.array(areas)
    minor = np.sum(areas < 0.005)
    moderate = np.sum((areas >= 0.005) & (areas < 0.025))
    severe = np.sum(areas >= 0.025)
    
    plt.figure(figsize=(10, 5))
    bars = plt.bar(["Minor (<0.005)", "Moderate (0.005-0.025)", "Severe (>=0.025)"], 
                   [minor, moderate, severe], 
                   color=["#2a9d8f", "#e76f51", "#d62828"], 
                   edgecolor="black", width=0.5)
    
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + 10, f"{yval:,} ({yval/len(areas)*100:.1f}%)", 
                 ha='center', va='bottom', fontweight='bold')
        
    plt.title(f"Ground-Truth Pothole Severity Distribution (Total: {len(areas):,} potholes)", fontsize=14, fontweight='bold')
    plt.ylabel("Count")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()
else:
    print("[NOTE] Run this cell once annotations are placed in 'dataset/'.")